## KLIP method after saving the intermediate measurement steps:
This notebook is an implementation of KLIP over PaDIS model and can be run over saved intermediate steps.

Update the paths in the configuration cell before running.

In [1]:
from pathlib import Path
import gc

import cv2
import numpy as np
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

# Configuration

In [2]:
CONFIG = {
    "padis_results_dir": Path("/data/akheirandish3/PaDIS_results/"),
    "sigma_folder": "Dps_24_calibration_aligned_new",
    "ood_folder_template": "Dps_24_NEW_OOD_{key_word}_full_intermediate_steps",
    "dataset_path": Path("/data2/CHAOS_test_ablation.npy"),
    "key_word": "default_medium",
    "run_id": 1,
    "sigma_power": 0.5,
    "patch_size": 2,
    "evaluation_windows": [(65, 85), (0, 100)],
}

# Helper functions

In [3]:
def load_all_artifacts_from_folder(
    base_path: Path,
    folder_name: str,
    run: int,
    data_type: str,
    pattern: str = "",
) -> dict[str, np.ndarray]:
    """
    Load all `.npy` artifacts matching `<artifact_id>_<data_type>.npy` from a run folder.
    """
    folder_path = Path(base_path) / folder_name / f"run_{run}"
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    suffix = f"_{data_type}.npy"
    artifact_ids = sorted(
        file_name[:-len(suffix)]
        for file_name in (p.name for p in folder_path.iterdir())
        if file_name.startswith(pattern) and file_name.endswith(suffix) and len(file_name) > len(suffix)
    )

    artifact_data: dict[str, np.ndarray] = {}
    for artifact_id in artifact_ids:
        file_path = folder_path / f"{artifact_id}_{data_type}.npy"
        artifact_data[artifact_id] = np.load(file_path)

    print(f"Loaded {len(artifact_data)} artifacts from {folder_path}")
    return artifact_data


def resize_image_batch(
    image_array: np.ndarray,
    target_size: tuple[int, int] = (256, 256),
    is_mask: bool = False,
) -> np.ndarray:
    """
    Resize a batch of 2D images from `(N, H, W)` to `(N, target_h, target_w)`.
    """
    if image_array.shape[1:3] == target_size[::-1]:
        return image_array.astype(np.uint8) if image_array.dtype == bool else image_array

    if image_array.dtype == bool:
        image_array = image_array.astype(np.uint8)

    interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    resized_images = [
        cv2.resize(img, target_size, interpolation=interpolation)
        for img in tqdm(image_array, desc=f"Resizing to {target_size}")
    ]
    return np.asarray(resized_images)


def downsample_mask_to_patch_grid(mask: np.ndarray, patch_size: int) -> np.ndarray:
    """
    Convert a pixel-level binary mask into a patch-level mask by averaging
    each non-overlapping patch and thresholding at > 0.
    """
    h, w = mask.shape
    if h % patch_size != 0 or w % patch_size != 0:
        raise ValueError("Mask size must be divisible by patch_size.")

    return (
        mask.reshape(h // patch_size, patch_size, w // patch_size, patch_size)
        .mean(axis=(1, 3)) > 0
    )


def compute_patch_energy_grid(
    artifact: np.ndarray,
    sigma_values: np.ndarray,
    window_start: int,
    window_end: int,
    patch_size: int,
    sigma_power: float = 0.5,
) -> np.ndarray:
    """
    Compute the average normalized patch energy over a time window.
    """
    if artifact.ndim != 4:
        raise ValueError(f"Expected artifact with shape (T, C, H, W), got {artifact.shape}")

    if not (0 <= window_start < window_end <= artifact.shape[0]):
        raise ValueError("Invalid time window for artifact length.")

    normalized = artifact[window_start:window_end, 0] / (
        sigma_values[window_start:window_end, None, None] ** sigma_power
    )

    t, h, w = normalized.shape
    if h % patch_size != 0 or w % patch_size != 0:
        raise ValueError("Image size must be divisible by patch_size.")

    patch_energy = (
        normalized.reshape(t, h // patch_size, patch_size, w // patch_size, patch_size) ** 2
    ).sum(axis=(0, 2, 4)) / t

    return patch_energy


def evaluate_window_auc(
    ood_artifacts: dict[str, np.ndarray],
    tumor_masks: np.ndarray,
    body_masks: np.ndarray,
    sigma_values: np.ndarray,
    window_start: int,
    window_end: int,
    patch_size: int,
    sigma_power: float = 0.5,
) -> float:
    """
    Evaluate mean ROC-AUC across all OOD artifacts for one time window.
    """
    auc_values: list[float] = []

    for artifact_id, artifact in ood_artifacts.items():
        # The notebook uses the second underscore-separated token as the image index.
        sample_idx = int(artifact_id.split("_")[1])

        tumor_patch_mask = downsample_mask_to_patch_grid(tumor_masks[sample_idx] > 0, patch_size)
        body_patch_mask = downsample_mask_to_patch_grid(body_masks[sample_idx] > 0, patch_size)

        score_map = compute_patch_energy_grid(
            artifact=artifact,
            sigma_values=sigma_values,
            window_start=window_start,
            window_end=window_end,
            patch_size=patch_size,
            sigma_power=sigma_power,
        )

        scores = score_map[body_patch_mask]
        labels = tumor_patch_mask[body_patch_mask]

        # Skip degenerate cases where ROC-AUC is undefined.
        if len(np.unique(labels)) < 2:
            continue

        fpr, tpr, _ = roc_curve(labels, scores)
        auc_values.append(auc(fpr, tpr))

    if not auc_values:
        raise RuntimeError("No valid samples were available for ROC-AUC computation.")

    return float(np.mean(auc_values))


def load_sigma_values(base_path: Path, sigma_folder: str, run_id: int) -> np.ndarray:
    """
    Load the first sigma array from the sigma folder.
    """
    sigma_artifacts = load_all_artifacts_from_folder(
        base_path=base_path,
        folder_name=sigma_folder,
        run=run_id,
        data_type="sigma_values",
    )
    sigma_values = next(iter(sigma_artifacts.values()), None)
    if sigma_values is None:
        raise RuntimeError("Failed to load sigma values.")
    return sigma_values


def load_ood_artifacts(base_path: Path, folder_name: str, run_id: int) -> dict[str, np.ndarray]:
    """
    Load measurement updates for the selected OOD folder.
    """
    return load_all_artifacts_from_folder(
        base_path=base_path,
        folder_name=folder_name,
        run=run_id,
        data_type="measurement_updates",
    )


def load_ablation_dataset(dataset_path: Path, key_word: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Load body and tumor masks from the ablation dataset.
    """
    data = np.load(dataset_path, allow_pickle=True).item()
    base_name = f"ood_tumor_{key_word}"

    body_masks = data[f"{base_name}_masks"][:, ::2, ::2]
    tumor_masks = data[f"{base_name}_labels"][:, ::2, ::2] > 1

    return body_masks, tumor_masks

# Load sigma, OOD artifacts, and dataset masks

In [4]:
key_word = CONFIG["key_word"]
ood_folder = CONFIG["ood_folder_template"].format(key_word=key_word)

sigma_values = load_sigma_values(
    base_path=CONFIG["padis_results_dir"],
    sigma_folder=CONFIG["sigma_folder"],
    run_id=CONFIG["run_id"],
)

ood_artifacts = load_ood_artifacts(
    base_path=CONFIG["padis_results_dir"],
    folder_name=ood_folder,
    run_id=CONFIG["run_id"],
)

body_masks, tumor_masks = load_ablation_dataset(
    dataset_path=CONFIG["dataset_path"],
    key_word=key_word,
)

print(f"Body mask shape:  {body_masks.shape}")
print(f"Tumor mask shape: {tumor_masks.shape}")

Loaded 705 artifacts from /data/akheirandish3/PaDIS_results/Dps_24_calibration_aligned_new/run_1
Loaded 100 artifacts from /data/akheirandish3/PaDIS_results/Dps_24_NEW_OOD_default_medium_full_intermediate_steps/run_1
Body mask shape:  (100, 256, 256)
Tumor mask shape: (100, 256, 256)


# Evaluate mean ROC-AUC for selected time windows

In [5]:
for window_start, window_end in CONFIG["evaluation_windows"]:
    mean_auc = evaluate_window_auc(
        ood_artifacts=ood_artifacts,
        tumor_masks=tumor_masks,
        body_masks=body_masks,
        sigma_values=sigma_values,
        window_start=window_start,
        window_end=window_end,
        patch_size=CONFIG["patch_size"],
        sigma_power=CONFIG["sigma_power"],
    )
    print(f"Window [{window_start}, {window_end}] -> mean AUC: {mean_auc:.6f}")

Window [65, 85] -> mean AUC: 0.672671
Window [0, 100] -> mean AUC: 0.596828


# Cleanup

In [6]:
gc.collect()

33